In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mPk import compute_matter_power_spectrum
import matplotlib as mpl

from classy_NEDE import Class as Class_NEDE

plt.rcParams.update(plt.rcParamsDefault)

mpl.rcParams.update({
    "text.usetex": False,
    "font.family": "serif",
    "font.serif": ["CMU Serif", "Computer Modern Roman", "DejaVu Serif"],
    "mathtext.fontset": "cm",
    "axes.unicode_minus": False,
    "font.size": 18,
    "axes.labelsize": 18,
    "axes.titlesize": 18,
    "legend.fontsize": 14,
    "axes.linewidth": 1.0,
    "lines.linewidth": 2.0,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.minor.visible": True,
    "ytick.minor.visible": True,
    "legend.frameon": False,
    "savefig.format": "pdf",
    "savefig.bbox": "tight",
    "xtick.top": True,
    "ytick.right": True,
})

In [ ]:
# ----------------------------
# 0) Common CLASS settings
# ----------------------------
zPk = "50.0,0"                      # redshifts at which to compute P(k)
k_targets = np.array([0.05, 1.0])   # 1/Mpc
kstr = ",".join(str(x) for x in k_targets[::-1])

base_dict = {
    'h': 0.6766,
    'omega_b': 0.02242,
    'omega_cdm': 0.1192,
    'A_s': 2.107699e-9,
    'n_s': 0.9671182,
    'output': 'dTk, mPk, vTk',
    'lensing': 'no',

    # neutrino sector (your baseline)
    'N_ur': 2.0328,
    'N_ncdm': 1,
    'deg_ncdm': 1,
    'm_ncdm': 0.06,
    'T_ncdm': 0.71611,

    'YHe': 0.245,

    'z_pk': zPk,
    'k_output_values': kstr,
    'P_k_max_1/Mpc': 15.0,
    'non_linear': 'halofit',
    'k_pivot': 0.05,

    'k_per_decade_for_pk': 100,
    'k_per_decade_for_bao': 200,
}

# ----------------------------
# 1) Baseline NEDE parameters
# ----------------------------
NEDE_params = {
    'l_switch_limber': 9,
    'delta_Neff_drmd': 0.83,
    'z_stop': 10**(4.83),
    'f_idm_drmd': 0.0285,
    'G_over_aH_drmd_ini': 10**(13.057),
}

# ----------------------------
# 2) 1σ definitions (self-consistent)
# ----------------------------
# For some parameters you typically want *additive* 1σ (like delta_Neff and f_idm).
# For others you often want *fractional* 1σ (≈10%) rather than adding a huge absolute number.
NEDE_sigma_add = {
    'delta_Neff_drmd': 0.462 / 2,   # additive 1σ (from paper)
    'f_idm_drmd': 0.0152 / 2,       # additive 1σ (from paper)
}


NEDE_sigma_frac = {
    'z_stop': 0.10,
    'G_over_aH_drmd_ini': 0.10,
}

# ----------------------------
# 3) LCDM "turn off DRMD" params
# ----------------------------
LCDM_params = {
    'delta_Neff_drmd': 0.0,
    'f_idm_drmd': 0.0,
    'G_over_aH_drmd_ini': 0.0,
    'z_stop': 0.0
}

# ----------------------------
# 4) Utilities: shifting params
# ----------------------------
def shifted_NEDE_params(sigma_shifts=None, raw_overrides=None):
    """
    Returns a dict of NEDE parameters after applying n-sigma shifts and raw overrides.

    sigma_shifts: dict like {'delta_Neff_drmd': +1, 'z_stop': -1, ...}
                  Interpretation:
                    - if key in NEDE_sigma_add: p <- p + n*sigma_add
                    - if key in NEDE_sigma_frac: p <- p * (1 + n*sigma_frac)
                  (So z_stop/G_over_aH vary multiplicatively.)
    raw_overrides: dict of exact values to set after sigma shifts (highest priority).
    """
    p = NEDE_params.copy()

    if sigma_shifts:
        for key, n in sigma_shifts.items():
            if key in NEDE_sigma_add:
                p[key] = p[key] + n * NEDE_sigma_add[key]
            elif key in NEDE_sigma_frac:
                frac = NEDE_sigma_frac[key]
                p[key] = p[key] * (1.0 + n * frac)
            else:
                raise KeyError(
                    f"'{key}' not configured in NEDE_sigma_add or NEDE_sigma_frac. "
                    f"Add it to one of them."
                )

    if raw_overrides:
        p.update(raw_overrides)

    return p

# ----------------------------
# 5) One consistent builder for NEDE and effective LCDM
# ----------------------------
def build_cosmo(model="NEDE", sigma_shifts=None, raw_overrides=None):
    """
    model: "NEDE" or "LCDM_eff"
      - "NEDE": uses NEDE sector directly (delta_Neff_drmd etc.)
      - "LCDM_eff": turns off DRMD sector AND maps delta_Neff_drmd -> extra N_ur

    sigma_shifts / raw_overrides refer to NEDE-parameter names.
    """
    d = base_dict.copy()

    # get the shifted NEDE parameter set (even if we build LCDM_eff, we still use it to decide N_ur)
    p = shifted_NEDE_params(sigma_shifts=sigma_shifts, raw_overrides=raw_overrides)

    if model == "NEDE":
        d.update(p)

    elif model == "LCDM_eff":
        # turn off DRMD sector
        d.update(LCDM_params)

        # map delta_Neff_drmd into N_ur (with the same shifted value as the NEDE case)
        d['N_ur'] = base_dict['N_ur'] + p['delta_Neff_drmd']

        # (optional) if your Class_NEDE complains about z_stop etc. when DRMD is off,
        # ensure they remain zero by not including them. We do NOT update d with p here.

    else:
        raise ValueError("model must be 'NEDE' or 'LCDM_eff'")

    cosmo = Class_NEDE()
    cosmo.set(d)
    cosmo.compute()
    return cosmo

In [ ]:
def write_R_and_halofit_mPk(cosmo_NEDE, cosmo_LCDM, filename_prefix):
    # Compute matter power spectra
    res_NEDE_z50 = compute_matter_power_spectrum(cosmo_NEDE, 50.0, nonlinear = True)
    res_LCDM_z50 = compute_matter_power_spectrum(cosmo_LCDM,  50.0, nonlinear = True)

    k_NEDE_z50 = res_NEDE_z50['k']
    mPk_NEDE_z50_nl = res_NEDE_z50['P_k']
    Tcb_NEDE_z50_nl = res_NEDE_z50['T_cb']

    k_LCDM_z50 = res_LCDM_z50['k']
    mPk_LCDM_z50_nl = res_LCDM_z50['P_k']
    Tcb_LCDM_z50_nl = res_LCDM_z50['T_cb']

    res_NEDE_z0 = compute_matter_power_spectrum(cosmo_NEDE, 0.0, nonlinear = True)
    k_NEDE_z0_nl = res_NEDE_z0['k']
    mPk_NEDE_z0_nl = res_NEDE_z0['P_k']

    # Compute R(k) = P_NEDE(k) / P_LCDM(k) at z=50
    R_k = Tcb_NEDE_z50_nl / Tcb_LCDM_z50_nl

    # Save to files
    np.savetxt(f"{filename_prefix}_R_k.txt", np.column_stack((k_NEDE_z50, R_k, Tcb_NEDE_z50_nl, Tcb_LCDM_z50_nl)), header="k [1/Mpc], R(k) = Tcb_NEDE(k)/Tcb_LCDM(k) @ z=50, Tcb_NEDE(k), Tcb_LCDM(k)")
    # np.savetxt(f"{filename_prefix}_halofit_mPk_NEDE_z0.txt", np.column_stack((k_NEDE_z0_nl, mPk_NEDE_z0_nl)), header="k [1/Mpc], P_NEDE(k) @ z=0 [Mpc^3]")

    return print(f"Written R(k) and halofit mPk to files with prefix '{filename_prefix}'.")


In [ ]:
# # Pure dNeff cosmologies

# # dNeff +0.5σ:
# NEDE_dNeff_p05 = build_cosmo("NEDE", sigma_shifts={'delta_Neff_drmd': +0.5})
# LCDM_dNeff_p05 = build_cosmo("LCDM_eff", sigma_shifts={'delta_Neff_drmd': +0.5})

# # dNeff -0.5σ:
# NEDE_dNeff_m05 = build_cosmo("NEDE", sigma_shifts={'delta_Neff_drmd': -0.5})
# LCDM_dNeff_m05 = build_cosmo("LCDM_eff", sigma_shifts={'delta_Neff_drmd': -0.5})

# # +1σ DeltaNeff:
# NEDE_dNeff_p1 = build_cosmo("NEDE",     sigma_shifts={'delta_Neff_drmd': +1})
# LCDM_dNeff_p1 = build_cosmo("LCDM_eff", sigma_shifts={'delta_Neff_drmd': +1})

# # -1σ DeltaNeff:
# NEDE_dNeff_m1 = build_cosmo("NEDE",     sigma_shifts={'delta_Neff_drmd': -1})
# LCDM_dNeff_m1 = build_cosmo("LCDM_eff", sigma_shifts={'delta_Neff_drmd': -1})

# # +1.5σ DeltaNeff:
# NEDE_dNeff_p15= build_cosmo("NEDE",     sigma_shifts={'delta_Neff_drmd': +1.5})
# LCDM_dNeff_p15 = build_cosmo("LCDM_eff", sigma_shifts={'delta_Neff_drmd': +1.5})

# # -1.5σ DeltaNeff:
# NEDE_dNeff_m15 = build_cosmo("NEDE",     sigma_shifts={'delta_Neff_drmd': -1.5})
# LCDM_dNeff_m15 = build_cosmo("LCDM_eff", sigma_shifts={'delta_Neff_drmd': -1.5})

# # +2σ DeltaNeff:
# NEDE_dNeff_p2 = build_cosmo("NEDE",     sigma_shifts={'delta_Neff_drmd': +2})
# LCDM_dNeff_p2 = build_cosmo("LCDM_eff", sigma_shifts={'delta_Neff_drmd': +2})

# # -2σ DeltaNeff:
# NEDE_dNeff_m2 = build_cosmo("NEDE",     sigma_shifts={'delta_Neff_drmd': -2})
# LCDM_dNeff_m2 = build_cosmo("LCDM_eff", sigma_shifts={'delta_Neff_drmd': -2})

In [ ]:
# # Pure f_idm cosmologies

# #f_idm +0.5σ:
# NEDE_fidm_p05 = build_cosmo("NEDE", sigma_shifts={'f_idm_drmd': +0.5})
# LCDM_fidm_p05 = build_cosmo("LCDM_eff", sigma_shifts={'f_idm_drmd': +0.5})  # won't change anything in LCDM_eff

# # f_idm -0.5σ:
# NEDE_fidm_m05 = build_cosmo("NEDE", sigma_shifts={'f_idm_drmd': -0.5})
# LCDM_fidm_m05 = build_cosmo("LCDM_eff", sigma_shifts={'f_idm_drmd': -0.5})  # won't change anything in LCDM_eff

# # +1σ f_idm
# NEDE_fidm_p1 = build_cosmo("NEDE", sigma_shifts={'f_idm_drmd': +1})
# LCDM_fidm_p1 = build_cosmo("LCDM_eff", sigma_shifts={'f_idm_drmd': +1})  # won't change anything in LCDM_eff

# # -1σ f_idm
# NEDE_fidm_m1 = build_cosmo("NEDE", sigma_shifts={'f_idm_drmd': -1})
# LCDM_fidm_m1 = build_cosmo("LCDM_eff", sigma_shifts={'f_idm_drmd': -1})  # won't change anything in LCDM_eff

# # +1.5σ f_idm
# NEDE_fidm_p15 = build_cosmo("NEDE", sigma_shifts={'f_idm_drmd': +1.5})
# LCDM_fidm_p15 = build_cosmo("LCDM_eff", sigma_shifts={'f_idm_drmd': +1.5})  # won't change anything in LCDM_eff

# # -1.5σ f_idm
# NEDE_fidm_m15 = build_cosmo("NEDE", sigma_shifts={'f_idm_drmd': -1.5})
# LCDM_fidm_m15 = build_cosmo("LCDM_eff", sigma_shifts={'f_idm_drmd': -1.5})  # won't change anything in LCDM_eff

# # +2σ f_idm
# NEDE_fidm_p2 = build_cosmo("NEDE", sigma_shifts={'f_idm_drmd': +2})
# LCDM_fidm_p2 = build_cosmo("LCDM_eff", sigma_shifts={'f_idm_drmd': +2})  # won't change anything in LCDM_eff

# # -2σ f_idm
# NEDE_fidm_m2 = build_cosmo("NEDE", sigma_shifts={'f_idm_drmd': -2})
# LCDM_fidm_m2 = build_cosmo("LCDM_eff", sigma_shifts={'f_idm_drmd': -2})  # won't change anything in LCDM_eff

In [ ]:
# # Mixed dNeff + f_idm cosmologies

# # +1σ dNeff +1σ f_idm
# NEDE_dNeff_p1_fidm_p1 = build_cosmo("NEDE", sigma_shifts={'delta_Neff_drmd': +1, 'f_idm_drmd': +1})
# LCDM_dNeff_p1_fidm_p1 = build_cosmo("LCDM_eff", sigma_shifts={'delta_Neff_drmd': +1, 'f_idm_drmd': +1})

# # -1σ dNeff -1σ f_idm
# NEDE_dNeff_m1_fidm_m1 = build_cosmo("NEDE", sigma_shifts={'delta_Neff_drmd': -1, 'f_idm_drmd': -1})
# LCDM_dNeff_m1_fidm_m1 = build_cosmo("LCDM_eff", sigma_shifts={'delta_Neff_drmd': -1, 'f_idm_drmd': -1})

# # +1σ dNeff -1σ f_idm
# NEDE_dNeff_p1_fidm_m1 = build_cosmo("NEDE", sigma_shifts={'delta_Neff_drmd': +1, 'f_idm_drmd': -1})
# LCDM_dNeff_p1_fidm_m1 = build_cosmo("LCDM_eff", sigma_shifts={'delta_Neff_drmd': +1, 'f_idm_drmd': -1})

# # -1σ dNeff +1σ f_idm
# NEDE_dNeff_m1_fidm_p1 = build_cosmo("NEDE", sigma_shifts={'delta_Neff_drmd': -1, 'f_idm_drmd': +1})
# LCDM_dNeff_m1_fidm_p1 = build_cosmo("LCDM_eff", sigma_shifts={'delta_Neff_drmd': -1, 'f_idm_drmd': +1})

# # +2σ dNeff +2σ f_idm
# NEDE_dNeff_p2_fidm_p2 = build_cosmo("NEDE", sigma_shifts={'delta_Neff_drmd': +2, 'f_idm_drmd': +2})
# LCDM_dNeff_p2_fidm_p2 = build_cosmo("LCDM_eff", sigma_shifts={'delta_Neff_drmd': +2, 'f_idm_drmd': +2})

# # -2σ dNeff -2σ f_idm
# NEDE_dNeff_m2_fidm_m2 = build_cosmo("NEDE", sigma_shifts={'delta_Neff_drmd': -2, 'f_idm_drmd': -2})
# LCDM_dNeff_m2_fidm_m2 = build_cosmo("LCDM_eff", sigma_shifts={'delta_Neff_drmd': -2, 'f_idm_drmd': -2})

# # +2σ dNeff -2σ f_idm
# NEDE_dNeff_p2_fidm_m2 = build_cosmo("NEDE", sigma_shifts={'delta_Neff_drmd': +2, 'f_idm_drmd': -2})
# LCDM_dNeff_p2_fidm_m2 = build_cosmo("LCDM_eff", sigma_shifts={'delta_Neff_drmd': +2, 'f_idm_drmd': -2})

# # -2σ dNeff +2σ f_idm
# NEDE_dNeff_m2_fidm_p2 = build_cosmo("NEDE", sigma_shifts={'delta_Neff_drmd': -2, 'f_idm_drmd': +2})
# LCDM_dNeff_m2_fidm_p2 = build_cosmo("LCDM_eff", sigma_shifts={'delta_Neff_drmd': -2, 'f_idm_drmd': +2})

In [ ]:
# # GaH +1dex and -1dex and +2dex and -2dex and +3dex and -3dex
# NEDE_GaH_p1dex = build_cosmo("NEDE", raw_overrides={'G_over_aH_drmd_ini': 10**(13.057) * 10**(1)})  # +1 dex
# LCDM_GaH_p1dex = build_cosmo("LCDM_eff", raw_overrides={'G_over_aH_drmd_ini': 10**(13.057) * 10**(1)})  # won't change anything in LCDM_eff

# NEDE_GaH_m1dex = build_cosmo("NEDE", raw_overrides={'G_over_aH_drmd_ini': 10**(13.057) * 10**(-1)})  # -1 dex
# LCDM_GaH_m1dex = build_cosmo("LCDM_eff", raw_overrides={'G_over_aH_drmd_ini': 10**(13.057) * 10**(-1)})  # won't change anything in LCDM_eff

# NEDE_GaH_p2dex = build_cosmo("NEDE", raw_overrides={'G_over_aH_drmd_ini': 10**(13.057) * 10**(2)})  # +2 dex
# LCDM_GaH_p2dex = build_cosmo("LCDM_eff", raw_overrides={'G_over_aH_drmd_ini': 10**(13.057) * 10**(2)})  # won't change anything in LCDM_eff

# NEDE_GaH_m2dex = build_cosmo("NEDE", raw_overrides={'G_over_aH_drmd_ini': 10**(13.057) * 10**(-2)})  # -2 dex
# LCDM_GaH_m2dex = build_cosmo("LCDM_eff", raw_overrides={'G_over_aH_drmd_ini': 10**(13.057) * 10**(-2)})  # won't change anything in LCDM_eff

# NEDE_GaH_p3dex = build_cosmo("NEDE", raw_overrides={'G_over_aH_drmd_ini': 10**(13.057) * 10**(3)})  # +3 dex
# LCDM_GaH_p3dex = build_cosmo("LCDM_eff", raw_overrides={'G_over_aH_drmd_ini': 10**(13.057) * 10**(3)})  # won't change anything in LCDM_eff

# NEDE_GaH_m3dex = build_cosmo("NEDE", raw_overrides={'G_over_aH_drmd_ini': 10**(13.057) * 10**(-3)})  # -3 dex
# LCDM_GaH_m3dex = build_cosmo("LCDM_eff", raw_overrides={'G_over_aH_drmd_ini': 10**(13.057) * 10**(-3)})  # won't change anything in LCDM_eff

In [ ]:
# # zstop +25% and -25% and +50% and -50%
# NEDE_zstop_p25 = build_cosmo("NEDE", raw_overrides={'z_stop': 10**(4.83) * 1.25})  # +25%
# LCDM_zstop_p25 = build_cosmo("LCDM_eff", raw_overrides={'z_stop': 10**(4.83) * 1.25})  # won't change anything in LCDM_eff
# NEDE_zstop_m25 = build_cosmo("NEDE", raw_overrides={'z_stop': 10**(4.83) * 0.75})  # -25%
# LCDM_zstop_m25 = build_cosmo("LCDM_eff", raw_overrides={'z_stop': 10**(4.83) * 0.75})  # won't change anything in LCDM_eff
# NEDE_zstop_p50 = build_cosmo("NEDE", raw_overrides={'z_stop': 10**(4.83) * 1.50})  # +50
# LCDM_zstop_p50 = build_cosmo("LCDM_eff", raw_overrides={'z_stop': 10**(4.83) * 1.50})  # won't change anything in LCDM_eff
# NEDE_zstop_m50 = build_cosmo("NEDE", raw_overrides={'z_stop': 10**(4.83) * 0.50})  # -50
# LCDM_zstop_m50 = build_cosmo("LCDM_eff", raw_overrides={'z_stop': 10**(4.83) * 0.50})  # won't change anything in LCDM_eff

In [ ]:
# Test cosmology to see if model works
# NEDE_test = build_cosmo("NEDE", sigma_shifts={'delta_Neff_drmd': 1.75, 'f_idm_drmd': -1.75})
# LCDM_test = build_cosmo("LCDM_eff", sigma_shifts={'delta_Neff_drmd': 1.75, 'f_idm_drmd': -1.75})

# NEDE_test2 = build_cosmo("NEDE", sigma_shifts={'delta_Neff_drmd': 0.5, 'f_idm_drmd': 0.5})
# LCDM_test2 = build_cosmo("LCDM_eff", sigma_shifts={'delta_Neff_drmd': 0.5, 'f_idm_drmd': 0.5})

# NEDE_test3 = build_cosmo("NEDE", sigma_shifts={'delta_Neff_drmd': 2.88, 'f_idm_drmd': 2.88})
# LCDM_test3 = build_cosmo("LCDM_eff", sigma_shifts={'delta_Neff_drmd': 2.88, 'f_idm_drmd': 2.88})

# NEDE_test4 = build_cosmo("NEDE", sigma_shifts={'delta_Neff_drmd': -0.25, 'f_idm_drmd': 2.0})
# LCDM_test4 = build_cosmo("LCDM_eff", sigma_shifts={'delta_Neff_drmd': -0.25, 'f_idm_drmd': 2.0})

# NEDE_test5 = build_cosmo("NEDE", sigma_shifts={'delta_Neff_drmd': -2.0, 'f_idm_drmd': 0.75})
# LCDM_test5 = build_cosmo("LCDM_eff", sigma_shifts={'delta_Neff_drmd': -2.0, 'f_idm_drmd': 0.75})

# NEDE_test6 = build_cosmo("NEDE", sigma_shifts={'delta_Neff_drmd': 2.5, 'f_idm_drmd': -2.5})
# LCDM_test6 = build_cosmo("LCDM_eff", sigma_shifts={'delta_Neff_drmd': 2.5, 'f_idm_drmd': -2.5})

# Planck cosmology
# LCDM_planck = build_cosmo("LCDM_eff", raw_overrides={'N_ur': 2.0328})

In [ ]:
# write_R_and_halofit_mPk(NEDE_test, LCDM_test, "test_dNeff_1.75_fidm-1.75")
# write_R_and_halofit_mPk(NEDE_test4, LCDM_test4, "test_dNeff_-0.25_fidm_2.0")
# write_R_and_halofit_mPk(NEDE_test5, LCDM_test5, "test_dNeff_-2.0_fidm_0.75")
# write_R_and_halofit_mPk(NEDE_test6, LCDM_test6, "test_dNeff_2.5_fidm_-2.5")

In [ ]:
# # Write to files for GaH and zstop variations
# write_R_and_halofit_mPk(NEDE_GaH_p1dex, LCDM_GaH_p1dex, "GaH_+1dex")
# write_R_and_halofit_mPk(NEDE_GaH_m1dex, LCDM_GaH_m1dex, "GaH_-1dex")
# write_R_and_halofit_mPk(NEDE_GaH_p2dex, LCDM_GaH_p2dex, "GaH_+2dex")
# write_R_and_halofit_mPk(NEDE_GaH_m2dex, LCDM_GaH_m2dex, "GaH_-2dex")
# write_R_and_halofit_mPk(NEDE_GaH_p3dex, LCDM_GaH_p3dex, "GaH_+3dex")
# write_R_and_halofit_mPk(NEDE_GaH_m3dex, LCDM_GaH_m3dex, "GaH_-3dex")

# write_R_and_halofit_mPk(NEDE_zstop_p25, LCDM_zstop_p25, "zstop_+2.5")
# write_R_and_halofit_mPk(NEDE_zstop_m25, LCDM_zstop_m25, "zstop_-2.5")
# write_R_and_halofit_mPk(NEDE_zstop_p50, LCDM_zstop_p50, "zstop_+5")
# write_R_and_halofit_mPk(NEDE_zstop_m50, LCDM_zstop_m50, "zstop_-5")

In [ ]:
# # # baseline
# write_R_and_halofit_mPk(NEDE_baseline, LCDM_baseline, "baseline")
# # dNeff +0.5σ
# write_R_and_halofit_mPk(NEDE_dNeff_p05, LCDM_dNeff_p05, "dNeff_+05")
# # dNeff -0.5σ
# write_R_and_halofit_mPk(NEDE_dNeff_m05, LCDM_dNeff_m05, "dNeff_-05")
# # +1σ dNeff
# write_R_and_halofit_mPk(NEDE_dNeff_p1, LCDM_dNeff_p1, "dNeff_+1")
# # -1σ dNeff
# write_R_and_halofit_mPk(NEDE_dNeff_m1, LCDM_dNeff_m1, "dNeff_-1")
# # +1.5σ dNeff
# write_R_and_halofit_mPk(NEDE_dNeff_p15, LCDM_dNeff_p15, "dNeff_+15")
# # -1.5σ dNeff
# write_R_and_halofit_mPk(NEDE_dNeff_m15, LCDM_dNeff_m15, "dNeff_-15")
# # +2σ dNeff
# write_R_and_halofit_mPk(NEDE_dNeff_p2, LCDM_dNeff_p2, "dNeff_+2")
# # -2σ dNeff
# write_R_and_halofit_mPk(NEDE_dNeff_m2, LCDM_dNeff_m2, "dNeff_-2")
# # f_idm +0.5σ
# write_R_and_halofit_mPk(NEDE_fidm_p05, LCDM_fidm_p05, "fidm_+05")
# # f_idm -0.5σ
# write_R_and_halofit_mPk(NEDE_fidm_m05, LCDM_fidm_m05, "fidm_-05")
# # +1σ f_idm
# write_R_and_halofit_mPk(NEDE_fidm_p1, LCDM_fidm_p1, "fidm_+1")
# # -1σ f_idm
# write_R_and_halofit_mPk(NEDE_fidm_m1, LCDM_fidm_m1, "fidm_-1")
# # +1.5σ f_idm
# write_R_and_halofit_mPk(NEDE_fidm_p15, LCDM_fidm_p15, "fidm_+15")
# # -1.5σ f_idm
# write_R_and_halofit_mPk(NEDE_fidm_m15, LCDM_fidm_m15, "fidm_-15")
# # +2σ f_idm
# write_R_and_halofit_mPk(NEDE_fidm_p2, LCDM_fidm_p2, "fidm_+2")
# # -2σ f_idm
# write_R_and_halofit_mPk(NEDE_fidm_m2, LCDM_fidm_m2, "fidm_-2")
# # # +1σ dNeff +1σ f_idm
# # write_R_and_halofit_mPk(NEDE_dNeff_p1_fidm_p1, LCDM_dNeff_p1_fidm_p1, "dNeff_+1_fidm_+1")
# # # -1σ dNeff -1σ f_idm
# # write_R_and_halofit_mPk(NEDE_dNeff_m1_fidm_m1, LCDM_dNeff_m1_fidm_m1, "dNeff_-1_fidm_-1")
# # # +1σ dNeff -1σ f_idm
# # write_R_and_halofit_mPk(NEDE_dNeff_p1_fidm_m1, LCDM_dNeff_p1_fidm_m1, "dNeff_+1_fidm_-1")
# # # -1σ dNeff +1σ f_idm
# # write_R_and_halofit_mPk(NEDE_dNeff_m1_fidm_p1, LCDM_dNeff_m1_fidm_p1, "dNeff_-1_fidm_+1")
# # # +2σ dNeff +2σ f_idm
# # write_R_and_halofit_mPk(NEDE_dNeff_p2_fidm_p2, LCDM_dNeff_p2_fidm_p2, "dNeff_+2_fidm_+2")
# # # -2σ dNeff -2σ f_idm
# # write_R_and_halofit_mPk(NEDE_dNeff_m2_fidm_m2, LCDM_dNeff_m2_fidm_m2, "dNeff_-2_fidm_-2")
# # # +2σ dNeff -2σ f_idm
# # write_R_and_halofit_mPk(NEDE_dNeff_p2_fidm_m2, LCDM_dNeff_p2_fidm_m2, "dNeff_+2_fidm_-2")
# # # -2σ dNeff +2σ f_idm
# # write_R_and_halofit_mPk(NEDE_dNeff_m2_fidm_p2, LCDM_dNeff_m2_fidm_p2, "dNeff_-2_fidm_+2")

In [ ]:
# # Write test cosmology
# write_R_and_halofit_mPk(NEDE_test5, LCDM_test5, "test_dNeff_-1.25")